# WhatsApp Chat Analysis: Exploratory Portfolio Notebook

This notebook demonstrates a reproducible WhatsApp chat analysis workflow using the modular `src/` pipeline. It covers:

- data loading and preprocessing
- temporal activity analysis
- Hinglish-aware transformer sentiment analysis
- weighted conversation network analysis
- export of research-ready outputs


## Methodology

1. Parse WhatsApp-exported text into a structured DataFrame.
2. Engineer temporal features such as date, hour, and session windows.
3. Run a transformer-based sentiment classifier on non-system messages.
4. Build a directed interaction graph using reply-gap weighting, mentions, and session clustering.
5. Export analysis tables to CSV and Parquet for downstream modeling or BI workflows.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

ROOT = Path.cwd().resolve().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from chat_analyser.config import load_config
from chat_analyser.pipeline import ChatAnalysisPipeline

sns.set_theme(style='whitegrid')

In [ ]:
config = load_config(ROOT / 'config' / 'defaults.json')
pipeline = ChatAnalysisPipeline(config)
results = pipeline.run_from_file(ROOT / 'data' / 'sample' / 'whatsapp_chat_sample.txt')

results.messages.head()

## Headline Stats

In [ ]:
results.stats

## Temporal Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results.daily_timeline['only_date'], results.daily_timeline['message_count'], marker='o', color='#ef476f')
ax.set_title('Daily Message Activity')
ax.set_ylabel('Messages')
plt.xticks(rotation=30)
plt.show()

## Top Words

In [ ]:
results.common_words.head(10)

## Sentiment Distribution

In [ ]:
results.sentiment_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(results.sentiment_summary['sentiment'].astype(str), results.sentiment_summary['messages'], color=['#d62828', '#fcbf49', '#2a9d8f'])
ax.set_title('Sentiment Distribution')
plt.show()

## Weighted Conversation Network

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
pos = nx.spring_layout(results.graph, seed=42, weight='weight')
edge_labels = {(u, v): f"{results.graph[u][v]['weight']:.1f}" for u, v in results.graph.edges}
nx.draw(results.graph, pos, with_labels=True, node_color='#a8dadc', node_size=1400, width=2.2, ax=ax)
nx.draw_networkx_edge_labels(results.graph, pos, edge_labels=edge_labels, ax=ax, font_size=8)
ax.set_title('Weighted Conversation Network')
ax.axis('off')
plt.show()

## Export Artifacts

In [ ]:
output_dir = ROOT / 'outputs' / 'notebook_run'
written_paths = pipeline.export(results, output_dir=output_dir)
written_paths[:8]

## Conclusions

- The modular pipeline makes it easy to move from app exploration to reproducible offline analysis.
- Transformer sentiment adds a richer signal than rule-based polarity, especially for code-mixed chat text.
- The weighted graph offers a more realistic interaction view than simple next-message chaining.
- Exported CSV/Parquet outputs make the project easy to extend into dashboards, experiments, or classical ML pipelines.
